In [1]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# 1. Load Model (Using 3B for speed and low VRAM)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\chibu\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0326 13:43:41.352867 29380 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.11: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 4050 Laptop GPU. Num GPUs = 1. Max memory: 5.997 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.8.0+cu129. CUDA: 8.9. CUDA Toolkit: 12.9. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 254/254 [00:05<00:00, 50.58it/s]
Unsloth: Will load unsloth/llama-3.2-3b-instruct-bnb-4bit as a legacy tokenizer.


In [2]:
# 2. Add LoRA Adapters (The standard efficient way)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

Unsloth 2026.3.11 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [3]:
import torch
print(f"Is CUDA available? {torch.cuda.is_available()}")
print(f"Device Name: {torch.cuda.get_device_name(0)}")

Is CUDA available? True
Device Name: NVIDIA GeForce RTX 4050 Laptop GPU


In [4]:

dataset = load_dataset("json", data_files="supabase_python_basics.jsonl", split="train")
# 3. Setup Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset, # Use the JSONL we created earlier
    dataset_text_field = "text", # Ensure your JSONL has a 'text' field or use a mapping function
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Small step count for initial testing
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)

Generating train split: 500 examples [00:00, 41663.06 examples/s]
Unsloth: Tokenizing ["text"]: 100%|██████████| 500/500 [00:00<00:00, 5738.99 examples/s]


In [ ]:
trainer.train()

# Save the LoRA adapters (lightweight)
model.save_pretrained("supabase_python_model") 
tokenizer.save_pretrained("supabase_python_model")


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
1,3.519741
2,3.714121
3,3.587211
4,3.366988
5,3.283862
6,2.736068
7,2.347965
8,1.860882
9,1.580394
10,1.475937


('supabase_python_model\\tokenizer_config.json',
 'supabase_python_model\\chat_template.jinja',
 'supabase_python_model\\tokenizer.json')

In [17]:
import torch
from unsloth import FastLanguageModel

# Switch to inference mode 

FastLanguageModel.for_inference(model) 

def ask_postgrest(question, schema_context):

    # matches  training format exactly.
    prompt = (
        f"### Instruction: Convert to Supabase Python query. {schema_context}\n"
        f"### Input: {question}\n"
        f"### Output: "
    )
    
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    # Use a specific generation config to avoid shape mismatches
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 64,
        use_cache = False,  
        pad_token_id = tokenizer.eos_token_id,
        temperature = 0.7,
        top_p = 0.9,
    )
    
    result = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    
    # Extract just the generated URL
    return result[0].split("### Output: ")[1].strip()


schema = "Tables: events(id, name); players(id, gamertag, rating, eventid); sets(id, winnerid)"
query = "What players have a rating of 1000 or more?"

try:
    print(f"Generated URL: {ask_postgrest(query, schema)}")
except Exception as e:
    print(f"Error: {e}")

Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated URL: supabase.table('players').select('*').gte('rating', 1000)  
### Supabase Python query: supabase.table('players').select('*').gte('rating', 1000)  
### supabase.table('players').select('*').gte('rating', 1000)  
### supabase


In [14]:
import json
import random
from pathlib import Path

random.seed(99)

path = Path("supabase_python_basics_500.jsonl")

with path.open("r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

existing = {row["text"] for row in rows}

instructions = [
    "Convert to Supabase Python query.",
    "Translate the request into a Supabase Python SDK query.",
    "Generate a Supabase Python command from the request.",
    "Turn the request into a Supabase Python query.",
]

schema_context = (
    "events(id, name, createdat); "
    "players(id, gamertag, created_at, eventid, rating); "
    "sets(id, player1_id, created_at, eventid, player2_id, winnerid, completed_at)"
)

event_values = [1, 2, 3, 4, 5, 10, 20, 50, 100]
rating_values = [800, 1000, 1200, 1300, 1400, 1500, 1600, 1700, 1800, 2000]
player_ids = [1, 2, 3, 4, 5, 10, 20, 50, 100, 200]
event_names = ["Genesis", "EVO", "The Big House", "Frostbite", "CEO", "Shine", "Combo Breaker"]
prefixes = ["Ma", "Za", "Co", "Am", "Hu", "Le", "Ax", "Pl"]

def format_row(user_input, output):
    instruction = random.choice(instructions)
    return {
        "text": (
            f"### Instruction: {instruction} Tables: {schema_context}.\n"
            f"### Input: {user_input}\n"
            f"### Output: Supabase Python: {output}"
        )
    }

extra_templates = []
for eid in event_values:
    for rating in rating_values:
        extra_templates.append((
            f"Show the top 7 players from event {eid} with rating at least {rating}.",
            f"supabase.table('players').select('*').eq('eventid', {eid}).gte('rating', {rating}).order('rating', desc=True).limit(7)"
        ))
        extra_templates.append((
            f"Return players from event {eid} with rating at most {rating}, ordered by rating ascending.",
            f"supabase.table('players').select('*').eq('eventid', {eid}).lte('rating', {rating}).order('rating')"
        ))

for pid in player_ids:
    extra_templates.append((
        f"Get sets where player {pid} appears as either player1 or player2.",
        f"supabase.table('sets').select('*').or_('player1_id.eq.{pid},player2_id.eq.{pid}')"
    ))
    extra_templates.append((
        f"Get completed sets where player {pid} was the winner.",
        f"supabase.table('sets').select('*').eq('winnerid', {pid}).not_.is_('completed_at', 'null')"
    ))

for pref in prefixes:
    extra_templates.append((
        f"Find players whose gamertag contains '{pref}' and return only id and gamertag.",
        f"supabase.table('players').select('id, gamertag').ilike('gamertag', '%{pref}%')"
    ))

for name in event_names:
    extra_templates.append((
        f"Get events whose name starts with '{name[0]}' and return id and name.",
        f"supabase.table('events').select('id, name').ilike('name', '{name[0]}%')"
    ))

for eid in event_values:
    extra_templates.append((
        f"Get the 15 most recent completed sets for event {eid}.",
        f"supabase.table('sets').select('*').eq('eventid', {eid}).not_.is_('completed_at', 'null').order('completed_at', desc=True).limit(15)"
    ))
    extra_templates.append((
        f"Get unfinished sets for event {eid}.",
        f"supabase.table('sets').select('*').eq('eventid', {eid}).is_('completed_at', 'null')"
    ))

for inp, out in extra_templates:
    row = format_row(inp, out)
    if row["text"] not in existing:
        rows.append(row)
        existing.add(row["text"])
    if len(rows) >= 500:
        break

with path.open("w", encoding="utf-8") as f:
    for row in rows[:500]:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Rewrote file with {len(rows[:500])} samples at {path}")

Rewrote file with 233 samples at supabase_python_basics_500.jsonl
